This notebook:

Creates speaker-independent train/test splits for the project.

Speaker-independent means entire speakers are held out for testing.
This prevents the model from memorizing speaker-specific voice traits
and forces it to generalize to unseen speakers.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [2]:
def create_speaker_split(df, test_fraction=0.2, random_state=42):
    """
    Split the dataset so that entire speakers are in either train or test, never both.

    Parameters
    ----------
    df : pd.DataFrame — must have 'speaker_id' and 'emotion' columns
    test_fraction : float — approximate fraction of data for testing (by speakers)
    random_state : int — for reproducibility

    Returns
    -------
    train_df : pd.DataFrame
    test_df : pd.DataFrame
    """
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_fraction, random_state=random_state)

    # Split by speaker groups
    train_idx, test_idx = next(splitter.split(df, groups=df["speaker_id"]))

    train_df = df.iloc[train_idx].copy().reset_index(drop=True)
    test_df = df.iloc[test_idx].copy().reset_index(drop=True)

    # Verify no speaker leakage
    train_speakers = set(train_df["speaker_id"].unique())
    test_speakers = set(test_df["speaker_id"].unique())
    overlap = train_speakers & test_speakers

    print(f"{'=' * 50}")
    print(f"Speaker-Independent Split")
    print(f"{'=' * 50}")
    print(f"Train: {len(train_df)} clips from {len(train_speakers)} speakers")
    print(f"Test:  {len(test_df)} clips from {len(test_speakers)} speakers")
    print(f"Speaker overlap: {len(overlap)} (should be 0)")
    print(f"\nTrain speakers: {sorted(train_speakers)}")
    print(f"Test speakers:  {sorted(test_speakers)}")

    # Show emotion distribution in each split
    print(f"\nEmotion distribution:")
    print(f"{'Emotion':<12} {'Train':>8} {'Test':>8} {'Train %':>8} {'Test %':>8}")
    print("-" * 48)
    for emotion in sorted(df["emotion"].unique()):
        n_train = len(train_df[train_df["emotion"] == emotion])
        n_test = len(test_df[test_df["emotion"] == emotion])
        pct_train = 100 * n_train / len(train_df) if len(train_df) > 0 else 0
        pct_test = 100 * n_test / len(test_df) if len(test_df) > 0 else 0
        print(f"{emotion:<12} {n_train:>8} {n_test:>8} {pct_train:>7.1f}% {pct_test:>7.1f}%")

    # Show gender distribution
    print(f"\nGender distribution:")
    for split_name, split_df in [("Train", train_df), ("Test", test_df)]:
        counts = split_df["gender"].value_counts().to_dict()
        print(f"  {split_name}: {counts}")

    print(f"{'=' * 50}")

    return train_df, test_df

In [3]:
def create_validation_split(train_df, val_fraction=0.15, random_state=42):
    """
    Further split the training set into train and validation, again speaker-independent.

    Parameters
    ----------
    train_df : pd.DataFrame — the training split from create_speaker_split
    val_fraction : float — fraction of training data for validation
    random_state : int

    Returns
    -------
    train_final_df : pd.DataFrame
    val_df : pd.DataFrame
    """
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=random_state)
    train_idx, val_idx = next(splitter.split(train_df, groups=train_df["speaker_id"]))

    train_final_df = train_df.iloc[train_idx].copy().reset_index(drop=True)
    val_df = train_df.iloc[val_idx].copy().reset_index(drop=True)

    print(f"\nValidation split (from training set):")
    print(f"  Train: {len(train_final_df)} clips, Val: {len(val_df)} clips")
    print(f"  Train speakers: {sorted(train_final_df['speaker_id'].unique())}")
    print(f"  Val speakers:   {sorted(val_df['speaker_id'].unique())}")

    return train_final_df, val_df

In [7]:
def save_splits(train_df, test_df, val_df=None, output_dir=""):
    """Save split DataFrames as CSV files for reproducibility."""

    train_path = os.path.join(output_dir, "train_split.csv")
    test_path = os.path.join(output_dir, "test_split.csv")
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
    print(f"\nSaved: {train_path} ({len(train_df)} rows)")
    print(f"Saved: {test_path} ({len(test_df)} rows)")

    if val_df is not None:
        val_path = os.path.join(output_dir, "val_split.csv")
        val_df.to_csv(val_path, index=False)
        print(f"Saved: {val_path} ({len(val_df)} rows)")

# Encode labels for model training

In [5]:
def encode_labels(df, emotion_column="emotion"):
    """
    Convert emotion strings to integer labels.

    Returns
    -------
    df : pd.DataFrame — with new 'label' column
    label_map : dict — emotion string → integer
    inverse_map : dict — integer → emotion string
    """
    emotions_sorted = sorted(df[emotion_column].unique())
    label_map = {emotion: idx for idx, emotion in enumerate(emotions_sorted)}
    inverse_map = {idx: emotion for emotion, idx in label_map.items()}

    df = df.copy()
    df["label"] = df[emotion_column].map(label_map)

    print(f"\nLabel encoding:")
    for emotion, idx in label_map.items():
        count = len(df[df["label"] == idx])
        print(f"  {idx}: {emotion} ({count} clips)")

    return df, label_map, inverse_map

In [8]:
# Load master DataFrame
df = pd.read_csv("master_with_features.csv")

# Create train/test split
train_df, test_df = create_speaker_split(df, test_fraction=0.2)

# Create validation split from training data
train_final_df, val_df = create_validation_split(train_df, val_fraction=0.15)

# Encode labels
train_final_df, label_map, inverse_map = encode_labels(train_final_df)
val_df["label"] = val_df["emotion"].map(label_map)
test_df["label"] = test_df["emotion"].map(label_map)

# Save everything
save_splits(train_final_df, test_df, val_df, output_dir="")

# Also save the label mapping
import json
with open("label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)
print(f"\nSaved label mapping to label_map.json")

Speaker-Independent Split
Train: 4148 clips from 24 speakers
Test:  380 clips from 6 speakers
Speaker overlap: 0 (should be 0)

Train speakers: ['ravdess_01', 'ravdess_02', 'ravdess_03', 'ravdess_04', 'ravdess_05', 'ravdess_06', 'ravdess_07', 'ravdess_08', 'ravdess_11', 'ravdess_12', 'ravdess_13', 'ravdess_14', 'ravdess_15', 'ravdess_17', 'ravdess_19', 'ravdess_20', 'ravdess_21', 'ravdess_22', 'ravdess_23', 'savee_DC', 'savee_JE', 'savee_JK', 'tess_OAF', 'tess_YAF']
Test speakers:  ['ravdess_09', 'ravdess_10', 'ravdess_16', 'ravdess_18', 'ravdess_24', 'savee_KL']

Emotion distribution:
Emotion         Train     Test  Train %   Test %
------------------------------------------------
angry             597       55    14.4%    14.5%
disgust           597       55    14.4%    14.5%
fearful           597       55    14.4%    14.5%
happy             597       55    14.4%    14.5%
neutral           566       50    13.6%    13.2%
sad               597       55    14.4%    14.5%
surprised      